# 03 曲线拟合的最小二乘法

曲线拟合通常面对含噪声数据。与插值不同，我们不要求曲线通过每个数据点，而是要求整体残差尽量小。

对模型

$$
p_n(x)=c_0+c_1x+\cdots+c_nx^n,
$$

在数据点 $x_i$ 上得到线性系统

$$
Ac\approx y,
$$

其中

$$
A_{ik}=x_i^k.
$$

离散最小二乘问题为

$$
\min_c \|Ac-y\|_2^2.
$$

几何上，$Ac$ 落在矩阵 $A$ 的列空间中。最小二乘解就是把数据向量 $y$ 正交投影到这个列空间上。


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["font.sans-serif"] = [
    "Arial Unicode MS",
    "PingFang SC",
    "Heiti SC",
    "SimHei",
    "Noto Sans CJK SC",
    "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False

ROOT = pathlib.Path.cwd()
while not (ROOT / "src" / "py_sc").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from py_sc import polynomial_eval, polynomial_least_squares


## 教学式实现：构造 Vandermonde 矩阵

下面先手写一个最小二乘拟合函数。它只保留核心步骤：

1. 构造升幂 Vandermonde 矩阵；
2. 调用 `np.linalg.lstsq` 求解最小二乘；
3. 返回升幂排列系数。

这里没有显式求解法方程，因为 $A^TA$ 会平方放大条件数。Notebook 中写教学版实现，是为了让矩阵结构和算法流程可见；`src/py_sc/approximation.py` 中的版本用于后续复用。


In [ ]:
def teaching_polynomial_least_squares(x, y, degree):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    A = np.vander(x, degree + 1, increasing=True)
    coefficients, *_ = np.linalg.lstsq(A, y, rcond=None)
    return coefficients

rng = np.random.default_rng(12)
x = np.linspace(-1.0, 1.0, 35)
y_clean = np.sin(2 * np.pi * x)
y_noisy = y_clean + 0.15 * rng.normal(size=x.size)

teaching_coefficients = teaching_polynomial_least_squares(x, y_noisy, degree=5)
formal_coefficients = polynomial_least_squares(x, y_noisy, degree=5)
print("教学版和正式版系数最大差异:", np.max(np.abs(teaching_coefficients - formal_coefficients)))


## 法方程与病态性

形式上，若 $A$ 满列秩，最小二乘解满足残差正交条件

$$
A^T(y-Ac)=0.
$$

整理得到法方程

$$
A^TAc=A^Ty.
$$

这个式子很好理解：最优残差必须垂直于 $A$ 的每一列。但数值计算中通常不推荐显式构造并求解法方程。原因是

$$
\kappa(A^TA)=\kappa(A)^2.
$$

对于幂基 Vandermonde 矩阵，高次多项式会使条件数快速变大。更稳健的做法是 QR 分解、SVD，或者像 `np.linalg.lstsq` 一样调用底层稳健线性代数例程。


In [ ]:
for degree in [3, 7, 12, 18]:
    A = np.vander(x, degree + 1, increasing=True)
    print(f"degree={degree:2d}, cond(A)={np.linalg.cond(A):.3e}")


## 过拟合实验

下面用同一组含噪声数据拟合不同次数的多项式。次数太低会欠拟合，次数太高则容易追随噪声并在端点附近振荡。

这说明训练残差小不等于模型好。一个高次多项式可以通过较大的系数穿过或贴近噪声点，但在未采样区域给出很差的预测。实际拟合时，需要同时考虑残差、模型复杂度、数值条件数和问题背景。


In [ ]:
x_eval = np.linspace(-1.0, 1.0, 600)
plt.figure(figsize=(8, 5))
plt.scatter(x, y_noisy, color="black", s=25, label="含噪声数据")
plt.plot(x_eval, np.sin(2 * np.pi * x_eval), color="gray", linewidth=2, label="真实函数")

for degree in [3, 7, 14]:
    coefficients = polynomial_least_squares(x, y_noisy, degree=degree)
    plt.plot(x_eval, polynomial_eval(coefficients, x_eval), label=f"{degree} 次拟合")

plt.title("多项式最小二乘拟合与过拟合")
plt.xlabel("x")
plt.ylabel("函数值")
plt.legend()
plt.tight_layout()
plt.show()


## 简单正则化思想

若希望抑制高次系数，可以考虑 Ridge/Tikhonov 形式：

$$
\min_c \|Ac-y\|_2^2+\lambda\|c\|_2^2.
$$

它对应线性方程

$$
(A^TA+\lambda I)c=A^Ty.
$$

其中 $\lambda>0$ 控制数据拟合和系数大小之间的折中。正则化会引入偏差，但可以降低模型对噪声和矩阵病态性的敏感程度。

需要注意：惩罚幂基系数的大小依赖于基函数选择。如果换成正交多项式基，正则化项的含义通常更稳定、更容易解释。本章第一轮只保留这个入口，后续可加入交叉验证和正则化路径实验。


In [ ]:
def teaching_ridge_fit(x, y, degree, lam):
    A = np.vander(x, degree + 1, increasing=True)
    lhs = A.T @ A + lam * np.eye(degree + 1)
    rhs = A.T @ y
    return np.linalg.solve(lhs, rhs)

ridge_coefficients = teaching_ridge_fit(x, y_noisy, degree=14, lam=1e-3)
plain_coefficients = polynomial_least_squares(x, y_noisy, degree=14)
print("普通 14 次拟合系数二范数:", np.linalg.norm(plain_coefficients))
print("Ridge 14 次拟合系数二范数:", np.linalg.norm(ridge_coefficients))


## 小结

* 离散最小二乘是拟合问题的基本形式。
* 最小二乘解可以看作把数据向量投影到设计矩阵的列空间。
* 法方程形式清楚，但数值上可能不稳定。
* 幂基 Vandermonde 矩阵在高次时容易病态。
* `np.linalg.lstsq` 比显式法方程更稳妥。
* 过高次数可能带来过拟合。
* 正则化是后续改善拟合稳定性的自然方向。
